In [8]:
# ==========================================
# CAR PRICE PREDICTION (ROBUST FINAL VERSION)
# ==========================================

import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv("data/car_data.csv")

# ==========================================
# CLEANING
# ==========================================

# Drop useless columns
df = df.drop(["Unnamed: 0", "car_name"], axis=1)

# Fix registration year (Jul-17 → 2017)
df["registration_year"] = df["registration_year"].str[-2:]
df["registration_year"] = df["registration_year"].astype(int) + 2000

# Extract numeric values safely
def extract_num(x):
    try:
        return float(str(x).split()[0])
    except:
        return np.nan

df["engine(cc)"] = df["engine(cc)"].apply(extract_num)
df["max_power(bhp)"] = df["max_power(bhp)"].apply(extract_num)
df["torque(Nm)"] = df["torque(Nm)"].apply(extract_num)

# Remove unrealistic values
df = df[df["torque(Nm)"] < 1000]

# ==========================================
# FEATURE ENGINEERING
# ==========================================

df["car_age"] = 2026 - df["manufacturing_year"]
df["kms_per_year"] = df["kms_driven"] / (df["car_age"] + 1)

df = df.drop(["manufacturing_year"], axis=1)

# ==========================================
# FEATURES & TARGET
# ==========================================

X = df.drop("price(in lakhs)", axis=1)
y = df["price(in lakhs)"]

# Log transform target (stabilizes variance)
y = np.log1p(y)

# Identify columns
categorical_cols = X.select_dtypes(include=["object"]).columns
numeric_cols = X.select_dtypes(exclude=["object"]).columns

# ==========================================
# PIPELINE (BEST PRACTICE)
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

model = RandomForestRegressor(random_state=42)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# ==========================================
# HYPERPARAMETER TUNING
# ==========================================

param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5],
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    n_jobs=-1,
    verbose=1
)

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==========================================
# TRAIN
# ==========================================

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

# ==========================================
# EVALUATION
# ==========================================

y_pred_log = best_model.predict(X_test)

# Convert back
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

print("\nBEST PARAMETERS:", grid.best_params_)
print("MAE:", mean_absolute_error(y_test_actual, y_pred))
print("R2 Score:", r2_score(y_test_actual, y_pred))

# ==========================================
# SAVE EVERYTHING
# ==========================================

with open("model/car_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("model/feature_columns.pkl", "wb") as f:
    pickle.dump(X.columns, f)

print("\nModel saved successfully!")

Fitting 3 folds for each of 12 candidates, totalling 36 fits

BEST PARAMETERS: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 200}
MAE: 528.1012356635622
R2 Score: 0.0355349549667795

Model saved successfully!
